# DynamicFrame vs Pure Spark — Handling Mixed-Type Data

## The Problem
You have orders coming from **3 different upstream systems** into the same S3 location:

| Source         | amount value | Type   | Reason                          |
|----------------|-------------|--------|---------------------------------|
| Mobile App     | `"N/A"`     | string | Form input, user left it blank  |
| Legacy ERP     | `199`       | int    | Old system, no decimal support  |
| New Backend    | `99.99`     | double | Properly typed                  |

All land in the same JSON file. You did NOT design this mess — the upstream teams did.

**Expected schema:** `amount` should be `double`

---
## The Data (o.json)
```json
{"order_id": "ORD-001", "customer_name": "john smith",   "amount": 99.99}   <- double  (New Backend)
{"order_id": "ORD-002", "customer_name": "jane doe",     "amount": "N/A"}   <- string  (Mobile App)
{"order_id": "ORD-003", "customer_name": "bob brown",    "amount": null}    <- null
{"order_id": "ORD-004", "customer_name": "alice green",  "amount": 199}     <- int     (Legacy ERP)
{"order_id": "ORD-005", "customer_name": "charlie black","amount": "150.75"}  <- String  (Mobile App)
```

---
## Table of Contents

**APPROACH 1 - Pure Spark**
- [Step 1A: Read with inferSchema](#Step-1A:-Read-with-inferSchema)
- [Step 1B: Cast amount to double](#Step-1B:-Cast-amount-to-double)
- [Step 1C: Enforce schema upfront](#Step-1C:-Enforce-schema-upfront)

**APPROACH 2 - Glue DynamicFrame**
- [Step 2A: Read - DynamicFrame detects mixed types](#Step-2A:-Read---DynamicFrame-detects-mixed-types)
- [Step 2B: resolveChoice - cast:double](#Step-2B:-resolveChoice---cast:double)
- [Step 2C: resolveChoice - project:double](#Step-2C:-resolveChoice---project:double)
- [Step 2D: resolveChoice - make_struct](#Step-2D:-resolveChoice---make_struct)
- [Step 2E: resolveChoice - make_cols](#Step-2E:-resolveChoice---make_cols)
- [Step 2F: Isolate bad records](#Step-2F:-Isolate-bad-records)

**Creating and Writing DynamicFrames**
- [Read using from_options](#Read-using-from_options)
- [Read using from_catalog](#Read-using-from_catalog)
- [Read using fromDF](#Read-using-fromDF)
- [Read using RDD to DynamicFrame](#Read-using-RDD-to-DynamicFrame)
- [Write using from_options](#Write-using-from_options)
- [Write using from_catalog](#Write-using-from_catalog)
- [Write using getSink](#Write-using-getSink)

**[Final Comparison](#Final-Comparison)**

---
## Setup

In [40]:
spark.stop()

In [39]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, upper, current_date
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql.types import StructType, StructField , IntegerType , StringType ,DoubleType

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

sc._jsc.hadoopConfiguration().set('fs.s3a.endpoint', 'http://minio:9000')
sc._jsc.hadoopConfiguration().set('fs.s3a.access.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.secret.key', 'minioadmin')
sc._jsc.hadoopConfiguration().set('fs.s3a.path.style.access', 'true')
sc._jsc.hadoopConfiguration().set('fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
sc._jsc.hadoopConfiguration().set('fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
sc._jsc.hadoopConfiguration().set('fs.s3a.connection.ssl.enabled', 'false')

print("spark session created")

S3_PATH = "s3a://glue-spark-etl-example-source/o.json"
# S3_PATH="s3://sanjay-de-bucket-2026/dynamic_frame_testing/o.json"
print("Setup complete")

spark session created
Setup complete


/home/glue_user/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [3]:
!aws s3 ls $S3_PATH

The code failed because of a fatal error. Some things to try: a) Make sure Spark has enough available resources for Jupyter to create a Spark context. b) Contact your Jupyter administrator to make sure the Spark magics library is configured correctly.   c) Restart the kernel.


---
# APPROACH 1 — Pure Spark

Spark must commit to a **single schema** at read time.
It scans the data and picks the widest compatible type — which is `string` here because `"N/A"` forces everything up to string.

### Step 1A: Read with inferSchema

In [35]:
df = spark.read \
    .json(S3_PATH)
df.printSchema()
df.show()

root
 |-- amount: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- order_id: string (nullable = true)

+-------+-------------+--------+
| amount|customer_name|order_id|
+-------+-------------+--------+
|  99.99|   john smith| ORD-001|
|    N/A|     jane doe| ORD-002|
|   null|    bob brown| ORD-003|
|    199|  alice green| ORD-004|
| 150.75|charlie black| ORD-005|
|hundred|       sanjay| ORD-006|
+-------+-------------+--------+



**Expected Output:**
```
root
 |-- amount: string (nullable = true)      <-- Spark promoted everything to string because of "N/A"
 |-- customer_name: string (nullable = true)
 |-- order_id: string (nullable = true)

+------+-------------+--------+
|amount|customer_name|order_id|
+------+-------------+--------+
| 99.99|   john smith| ORD-001|
|   N/A|     jane doe| ORD-002|
|  null|    bob brown| ORD-003|
|   199|  alice green| ORD-004|
|150.75|charlie black| ORD-005|
+------+-------------+--------+
```
> **Problem:** `amount` is `string`. You cannot do `SUM(amount)` or `AVG(amount)` directly. You must cast it manually.

### Step 1B: Cast amount to double

In [8]:
df.select(
    col("order_id"),
    col("customer_name"),
    col("amount").cast("double").alias("amount")
).show()

+--------+-------------+------+
|order_id|customer_name|amount|
+--------+-------------+------+
| ORD-001|   john smith| 99.99|
| ORD-002|     jane doe|  null|
| ORD-003|    bob brown|  null|
| ORD-004|  alice green| 199.0|
| ORD-005|charlie black|150.75|
| ORD-006|       sanjay|  null|
+--------+-------------+------+



**Expected Output:**
```
+--------+-------------+------+
|order_id|customer_name|amount|
+--------+-------------+------+
| ORD-001|   john smith| 99.99|
| ORD-002|     jane doe|  null|   <-- "N/A" silently became null. No error, no warning.
| ORD-003|    bob brown|  null|
| ORD-004|  alice green| 199.0|
| ORD-005|charlie black|150.75|
+--------+-------------+------+
```
> **Problem:** `"N/A"` silently became `null`. Spark did NOT tell you this happened.
> In production with millions of rows, you would never know how many records were silently lost.

### Step 1C: Enforce schema upfront

In [9]:
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("amount", DoubleType(), True)
])

df_enforced = spark.read.schema(schema).json(S3_PATH)
df_enforced.printSchema()
df_enforced.show()

root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- amount: double (nullable = true)

+--------+-------------+------+
|order_id|customer_name|amount|
+--------+-------------+------+
| ORD-001|   john smith| 99.99|
| ORD-002|     jane doe|  null|
| ORD-003|    bob brown|  null|
| ORD-004|  alice green| 199.0|
| ORD-005|charlie black|  null|
| ORD-006|       sanjay|  null|
+--------+-------------+------+



**Expected Output:**
```
root
 |-- order_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- amount: double (nullable = true)

+--------+-------------+------+
|order_id|customer_name|amount|
+--------+-------------+------+
| ORD-001|   john smith| 99.99|
| ORD-002|     jane doe|  null|   <-- "N/A" silently became null again. Same problem.
| ORD-003|    bob brown|  null|
| ORD-004|  alice green| 199.0|
| ORD-005|charlie black|150.75|
+--------+-------------+------+
```
> **Still the same problem:** Even with an enforced schema, Spark silently converts `"N/A"` to `null`.
> You have no way to separate the "intentionally null" rows from the "bad data" rows.

### Spark Summary
| Scenario | What Spark does | Your visibility |
|---|---|---|
| `inferSchema=true` | Promotes all to `string` | You see the raw mess |
| `.cast("double")` | `"N/A"` → `null` silently | **Zero visibility** |
| Enforced schema | `"N/A"` → `null` silently | **Zero visibility** |

**The core Spark limitation: it cannot distinguish between a legitimately null value and a bad/corrupt value.**

---
# APPROACH 2 — Glue DynamicFrame

DynamicFrame does NOT force a single type. It reads the data as-is and represents mixed types as a `choice` type.
You then **explicitly decide** what to do with each variant.

### Step 2A: Read - DynamicFrame detects mixed types

In [36]:
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": [S3_PATH]},
    format="json"
)
dyf.printSchema()
dyf.show()

root
|-- order_id: string
|-- customer_name: string
|-- amount: choice
|    |-- double
|    |-- int
|    |-- string

{"order_id": "ORD-001", "customer_name": "john smith", "amount": 99.99}
{"order_id": "ORD-002", "customer_name": "jane doe", "amount": "N/A"}
{"order_id": "ORD-003", "customer_name": "bob brown", "amount": null}
{"order_id": "ORD-004", "customer_name": "alice green", "amount": 199}
{"order_id": "ORD-005", "customer_name": "charlie black", "amount": "150.75"}
{"order_id": "ORD-006", "customer_name": "sanjay", "amount": "hundred"}


In [17]:
csv_path="s3a://glue-spark-etl-example-source/o.csv"
dyf = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={
        "paths": [csv_path]
    },
    format="csv",
    format_options={
        "withHeader": True,
        "separator": ","
    }
)
dyf.printSchema()
dyf.show()

root
|-- order_id: string
|-- customer_name: string
|-- amount: string

{"order_id": "ORD-001", "customer_name": "johnsmith", "amount": "99.99"}
{"order_id": "ORD-002", "customer_name": "janedoe", "amount": "N/A"}
{"order_id": "ORD-003", "customer_name": "bobbrown", "amount": "null"}
{"order_id": "ORD-004", "customer_name": "alicegreen", "amount": "199.50"}


**Expected Output:**
```
root
|-- order_id: string
|-- customer_name: string
|-- amount: choice          <-- DynamicFrame says: "I found multiple types here, YOU decide"
|    |-- double
|    |-- int
|    |-- string

{"order_id": "ORD-001", "customer_name": "john smith", "amount": 99.99}
{"order_id": "ORD-002", "customer_name": "jane doe",   "amount": "N/A"}
{"order_id": "ORD-003", "customer_name": "bob brown",  "amount": null}
{"order_id": "ORD-004", "customer_name": "alice green","amount": 199}
{"order_id": "ORD-005", "customer_name": "charlie black", "amount": "150.75"}
```
> **Key difference:** DynamicFrame did NOT crash. It did NOT silently drop data.
> It preserved every value exactly as it came in and flagged the ambiguity for you.

### Step 2B: resolveChoice - cast:double

In [37]:
resolved_cast = dyf.resolveChoice(specs=[('amount', 'cast:double')])
resolved_cast.printSchema()
resolved_cast.show()

root
|-- order_id: string
|-- customer_name: string
|-- amount: double

{"order_id": "ORD-001", "customer_name": "john smith", "amount": 99.99}
{"order_id": "ORD-002", "customer_name": "jane doe"}
{"order_id": "ORD-003", "customer_name": "bob brown"}
{"order_id": "ORD-004", "customer_name": "alice green", "amount": 199.0}
{"order_id": "ORD-005", "customer_name": "charlie black", "amount": 150.75}
{"order_id": "ORD-006", "customer_name": "sanjay"}


**Expected Output:**
```
root
|-- order_id: string
|-- customer_name: string
|-- amount: double

{"order_id": "ORD-001", "customer_name": "john smith",    "amount": 99.99}
{"order_id": "ORD-002", "customer_name": "jane doe"}                        <-- "N/A" became null, field omitted
{"order_id": "ORD-003", "customer_name": "bob brown"}                       <-- was already null
{"order_id": "ORD-004", "customer_name": "alice green",   "amount": 199.0}
{"order_id": "ORD-005", "customer_name": "charlie black", "amount": 150.75}
```
> Same null result as Spark for `"N/A"`, BUT you made this choice **consciously** after seeing the `choice` schema.
> You knew what you were dealing with before deciding.

### Step 2C: resolveChoice - project:double

In [12]:
resolved_project = dyf.resolveChoice(specs=[('amount', 'project:double')])
resolved_project.printSchema()
resolved_project.show()

root
|-- order_id: string
|-- customer_name: string
|-- amount: double

{"order_id": "ORD-001", "customer_name": "john smith", "amount": 99.99}
{"order_id": "ORD-002", "customer_name": "jane doe"}
{"order_id": "ORD-003", "customer_name": "bob brown"}
{"order_id": "ORD-004", "customer_name": "alice green"}
{"order_id": "ORD-005", "customer_name": "charlie black"}
{"order_id": "ORD-006", "customer_name": "sanjay"}


**Expected Output:**
```
root
|-- order_id: string
|-- customer_name: string
|-- amount: double

{"order_id": "ORD-001", "customer_name": "john smith",    "amount": 99.99}  <-- was double, kept
{"order_id": "ORD-002", "customer_name": "jane doe"}                        <-- was string, amount dropped
{"order_id": "ORD-003", "customer_name": "bob brown"}                       <-- was null, amount dropped
{"order_id": "ORD-004", "customer_name": "alice green"}                     <-- was int, amount dropped
{"order_id": "ORD-005", "customer_name": "charlie black"}                   <-- was string "150.75", dropped
```
> Only the rows where `amount` was natively a `double` are kept. Everything else is dropped from that field.
> Spark has no equivalent of this — it cannot distinguish `int 199` from `double 99.99` after reading.

### Step 2D: resolveChoice - make_struct

In [15]:
resolved_struct = dyf.resolveChoice(specs=[('amount', 'make_struct')])
resolved_struct.printSchema()
resolved_struct.show()
# Write as Parquet to S3
glueContext.write_dynamic_frame.from_options(
    frame=resolved_struct,
    connection_type="s3",
    connection_options={
        "path": "s3://glue-spark-etl-example-target/parquet_struct"
    },
    format="parquet"
)

root
|-- order_id: string
|-- customer_name: string
|-- amount: struct
|    |-- double: double
|    |-- int: int
|    |-- string: string

{"order_id": "ORD-001", "customer_name": "john smith", "amount": {"double": 99.99}}
{"order_id": "ORD-002", "customer_name": "jane doe", "amount": {"string": "N/A"}}
{"order_id": "ORD-003", "customer_name": "bob brown"}
{"order_id": "ORD-004", "customer_name": "alice green", "amount": {"int": 199}}
{"order_id": "ORD-005", "customer_name": "charlie black", "amount": {"string": "150.75"}}
{"order_id": "ORD-006", "customer_name": "sanjay", "amount": {"string": "hundred"}}


**Expected Output:**
```
root
|-- order_id: string
|-- customer_name: string
|-- amount: struct
|    |-- double: double
|    |-- int: int
|    |-- string: string

{"order_id": "ORD-001", "customer_name": "john smith",    "amount": {"double": 99.99}}
{"order_id": "ORD-002", "customer_name": "jane doe",      "amount": {"string": "N/A"}}
{"order_id": "ORD-003", "customer_name": "bob brown"}
{"order_id": "ORD-004", "customer_name": "alice green",   "amount": {"int": 199}}
{"order_id": "ORD-005", "customer_name": "charlie black", "amount": {"string": "150.75"}}
```
> This is the most powerful option for **auditing**. You can now see exactly which source sent which type.
> Spark cannot do this at all.

### Step 2E: resolveChoice - make_cols

Instead of collapsing all types into one column or wrapping them in a struct, `make_cols` **creates a separate column for each type found**.

So if `amount` had `double`, `int`, and `string` variants, you get:
```
amount_double   (contains value if original was double, else null)
amount_int      (contains value if original was int, else null)
amount_string   (contains value if original was string, else null)
```
Each row will have a value in exactly ONE of these columns and null in the others.

In [ ]:
resolved_cols = dyf.resolveChoice(specs=[('amount', 'make_cols')])
resolved_cols.printSchema()
resolved_cols.show()

**Expected Output:**
```
root
|-- order_id: string
|-- customer_name: string
|-- amount_double: double    <-- new column for double values
|-- amount_int: int          <-- new column for int values
|-- amount_string: string    <-- new column for string values

{"order_id": "ORD-001", "customer_name": "john smith",    "amount_double": 99.99}
{"order_id": "ORD-002", "customer_name": "jane doe",      "amount_string": "N/A"}
{"order_id": "ORD-003", "customer_name": "bob brown"}
{"order_id": "ORD-004", "customer_name": "alice green",   "amount_int": 199}
{"order_id": "ORD-005", "customer_name": "charlie black", "amount_string": "150.75"}
```
> Each row has a value in exactly ONE column. The other two are null for that row.

### make_cols vs make_struct — what is the difference?

| | `make_struct` | `make_cols` |
|---|---|---|
| Output | One column of type `struct` | Multiple flat columns |
| Schema | `amount: struct<double, int, string>` | `amount_double`, `amount_int`, `amount_string` |
| Athena query | `SELECT amount.double FROM ...` | `SELECT amount_double FROM ...` |
| Best for | Keeping data nested, writing to Parquet | Flat tables, Athena/Redshift queries |

**make_cols is easier to query in Athena** because the columns are flat — no dot notation needed.

```sql
-- make_struct: need dot notation
SELECT order_id, amount.double FROM orders WHERE amount.double IS NOT NULL

-- make_cols: plain column name
SELECT order_id, amount_double FROM orders WHERE amount_double IS NOT NULL
```

### Step 2F: Isolate bad records

In [ ]:
# After cast:double, rows where amount could not be cast are tracked as errors
resolved_cast = dyf.resolveChoice(specs=[('amount', 'cast:double')])

# Good records — amount is a valid double
good_df = resolved_cast.toDF().filter(col("amount").isNotNull())
print("=== GOOD RECORDS (send to data warehouse) ===")
good_df.show()

# Bad records — amount could not be cast (was "N/A" or null)
bad_df = resolved_cast.toDF().filter(col("amount").isNull())
print("=== BAD RECORDS (send to error/quarantine table) ===")
bad_df.show()

/home/glue_user/spark/python/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


=== GOOD RECORDS (send to data warehouse) ===
+--------+-------------+------+
|order_id|customer_name|amount|
+--------+-------------+------+
| ORD-001|   john smith| 99.99|
| ORD-004|  alice green| 199.0|
| ORD-005|charlie black|150.75|
+--------+-------------+------+

=== BAD RECORDS (send to error/quarantine table) ===
+--------+-------------+------+
|order_id|customer_name|amount|
+--------+-------------+------+
| ORD-002|     jane doe|  null|
| ORD-003|    bob brown|  null|
+--------+-------------+------+



**Expected Output:**
```
=== GOOD RECORDS (send to data warehouse) ===
+--------+-------------+------+
|order_id|customer_name|amount|
+--------+-------------+------+
| ORD-001|   john smith| 99.99|
| ORD-004|  alice green| 199.0|
| ORD-005|charlie black|150.75|
+--------+-------------+------+

=== BAD RECORDS (send to error/quarantine table) ===
+--------+-------------+------+
|order_id|customer_name|amount|
+--------+-------------+------+
| ORD-002|     jane doe|  null|   <-- came in as "N/A"
| ORD-003|    bob brown|  null|   <-- came in as null
+--------+-------------+------+
```
> This is the **production pattern**: good data goes to the warehouse, bad data goes to a quarantine table for the data team to investigate.
> With pure Spark, both rows just silently become `null` in the same table — you cannot separate them.

---
# Creating and Writing DynamicFrames

There are 2 main ways to create a DynamicFrame and 2 corresponding ways to write it back.

## Read using from_options

Use this when reading raw files from S3 directly without going through Glue Catalog.

In [ ]:
# Read CSV from S3
dyf_csv = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={
        "paths": ["s3://my-bucket/raw/orders/"],
        "recurse": True   # read all files in subfolders too
    },
    format="csv",
    format_options={"withHeader": True, "separator": ","}
)
dyf_csv.printSchema()
dyf_csv.show(5)

**Expected Output:**
```
root
|-- order_id: string
|-- order_date: string
|-- order_customer_id: string
|-- order_status: string

{"order_id": "1", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "11599", "order_status": "CLOSED"}
{"order_id": "2", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "256",   "order_status": "PENDING_PAYMENT"}
```

In [ ]:
# Read JSON from S3
dyf_json = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": ["s3://my-bucket/raw/orders/"]},
    format="json"
)
dyf_json.printSchema()
dyf_json.show(5)

In [ ]:
# Read Parquet from S3
dyf_parquet = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": ["s3://my-bucket/raw/orders/"]},
    format="parquet"
)
dyf_parquet.printSchema()
dyf_parquet.show(5)

## Read using from_catalog

Use this when the table is already registered in Glue Catalog (by a crawler or DDL).

In [ ]:
# Basic read from catalog
dyf_catalog = glueContext.create_dynamic_frame.from_catalog(
    database="ecommerce_db",
    table_name="orders"
)
dyf_catalog.printSchema()
dyf_catalog.show(5)

In [ ]:
# Read from catalog with push_down_predicate
# Filters at S3 level before loading into Spark — much faster than WHERE in spark.sql
dyf_filtered = glueContext.create_dynamic_frame.from_catalog(
    database="ecommerce_db",
    table_name="orders",
    push_down_predicate="order_status = 'COMPLETE'"
)
print(f"Record count: {dyf_filtered.count()}")
dyf_filtered.show(5)

**Expected Output:**
```
Record count: 3  <-- only COMPLETE records loaded, Glue never read the rest from S3

{"order_id": "3", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "12111", "order_status": "COMPLETE"}
{"order_id": "5", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "11318", "order_status": "COMPLETE"}
{"order_id": "6", "order_date": "2013-07-25 00:00:00.0", "order_customer_id": "7130",  "order_status": "COMPLETE"}
```

### from_options vs from_catalog

| | `from_options` | `from_catalog` |
|---|---|---|
| Where is table metadata? | You provide path + format in code | Stored in Glue Catalog |
| Need a crawler first? | No | Yes (or manual DDL) |
| push_down_predicate? | No | Yes |
| Job Bookmark support? | Yes | Yes |
| Best for | Ad-hoc reads, raw files | Production pipelines with catalog |


## Write using from_options

In [ ]:
# Write to S3 as Parquet
glueContext.write_dynamic_frame.from_options(
    frame=dyf_catalog,
    connection_type="s3",
    connection_options={"path": "s3://my-bucket/silver/orders/"},
    format="parquet"
)

In [ ]:
# Write to S3 as Parquet with partitioning
# Creates subfolders: order_status=COMPLETE/, order_status=CLOSED/ etc.
glueContext.write_dynamic_frame.from_options(
    frame=dyf_catalog,
    connection_type="s3",
    connection_options={
        "path": "s3://my-bucket/silver/orders/",
        "partitionKeys": ["order_status"]
    },
    format="parquet"
)

**S3 folder structure after partitioned write:**
```
s3://my-bucket/silver/orders/
    order_status=COMPLETE/
        part-00000.parquet
    order_status=CLOSED/
        part-00000.parquet
    order_status=PENDING_PAYMENT/
        part-00000.parquet
```
> When you query `WHERE order_status = 'COMPLETE'` in Athena, it only reads the `COMPLETE/` folder and skips the rest.

In [ ]:
# Write to S3 as CSV
glueContext.write_dynamic_frame.from_options(
    frame=dyf_catalog,
    connection_type="s3",
    connection_options={"path": "s3://my-bucket/silver/orders_csv/"},
    format="csv",
    format_options={"writeHeader": True, "separator": ","}
)

## Write using from_catalog

Use this when the target table is already registered in Glue Catalog.

In [ ]:
glueContext.write_dynamic_frame.from_catalog(
    frame=dyf_catalog,
    database="ecommerce_db",
    table_name="orders_silver"   # target table must already exist in catalog
)

### write from_options vs from_catalog

| | `write from_options` | `write from_catalog` |
|---|---|---|
| Target location | You specify S3 path in code | Taken from Glue Catalog table definition |
| Updates catalog metadata? | No | Yes |
| Target table must exist in catalog? | No | Yes |
| Partitioning | Via `partitionKeys` | Defined in catalog table |
| Best for | Writing to new S3 locations | Writing to existing catalog tables |

**When to use `df.write` instead:**
```python
# If you don't need catalog updates or Glue-specific options, native Spark is simpler
df.write.mode("append").parquet("s3://my-bucket/silver/orders/")
df.write.mode("append").partitionBy("order_status").parquet("s3://my-bucket/silver/orders/")
```

## Read using fromDF

Use this when you already have a Spark DataFrame and need to convert it to a DynamicFrame.

In [ ]:
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql.functions import col

# Start with a Spark DataFrame
df = spark.read.option("header", True).csv("s3://my-bucket/raw/orders/orders.csv")

# Do all your transformations in Spark
df_clean = df.filter(col("order_status") == "COMPLETE") \
              .withColumnRenamed("order_customer_id", "customer_id")

# Convert to DynamicFrame when you need Glue write options
# Arguments: (dataframe, glueContext, name_label)
dyf_from_df = DynamicFrame.fromDF(df_clean, glueContext, "orders_clean")

dyf_from_df.printSchema()
print(f"Count: {dyf_from_df.count()}")

**Expected Output:**
```
root
|-- order_id: string
|-- order_date: string
|-- customer_id: string    <-- renamed from order_customer_id
|-- order_status: string

Count: 3   <-- only COMPLETE records
```
> `fromDF()` does NOT copy data. It just wraps the existing DataFrame in a DynamicFrame shell.
> The third argument `"orders_clean"` is just an internal label — any string works.

## Read using RDD to DynamicFrame

Use this when your data starts as a raw RDD.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from awsglue.dynamicframe import DynamicFrame

# Step 1 — create raw data as RDD
raw_data = [
    ("1", "2013-07-25 00:00:00.0", "11599", "CLOSED"),
    ("2", "2013-07-25 00:00:00.0", "256",   "PENDING_PAYMENT"),
    ("3", "2013-07-25 00:00:00.0", "12111", "COMPLETE"),
    ("4", "2013-07-25 00:00:00.0", "8827",  "CLOSED"),
    ("5", "2013-07-25 00:00:00.0", "11318", "COMPLETE"),
    ("6", "2013-07-25 00:00:00.0", "7130",  "COMPLETE"),
]
rdd = sc.parallelize(raw_data)

# Step 2 — RDD → DataFrame with schema
schema = StructType([
    StructField("order_id",          StringType(), True),
    StructField("order_date",        StringType(), True),
    StructField("order_customer_id", StringType(), True),
    StructField("order_status",      StringType(), True),
])
df_from_rdd = spark.createDataFrame(rdd, schema)
df_from_rdd.show()

# Step 3 — DataFrame → DynamicFrame
dyf_from_rdd = DynamicFrame.fromDF(df_from_rdd, glueContext, "orders_from_rdd")
dyf_from_rdd.printSchema()
print(f"Count: {dyf_from_rdd.count()}")

**Expected Output:**
```
+--------+--------------------+-----------------+---------------+
|order_id|          order_date|order_customer_id|   order_status|
+--------+--------------------+-----------------+---------------+
|       1|2013-07-25 00:00:...|            11599|         CLOSED|
|       2|2013-07-25 00:00:...|              256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:...|            12111|       COMPLETE|
|       4|2013-07-25 00:00:...|             8827|         CLOSED|
|       5|2013-07-25 00:00:...|            11318|       COMPLETE|
|       6|2013-07-25 00:00:...|             7130|       COMPLETE|
+--------+--------------------+-----------------+---------------+

root
|-- order_id: string
|-- order_date: string
|-- order_customer_id: string
|-- order_status: string

Count: 6
```
> The full path is: **RDD → `spark.createDataFrame()` → DataFrame → `DynamicFrame.fromDF()` → DynamicFrame**
> You always go through DataFrame first. There is no direct RDD → DynamicFrame conversion.

## Write using getSink

Older Glue write API, auto-generated by Glue Visual Editor. Splits into getSink() + writeFrame().

In [ ]:
# Write to S3 using getSink — equivalent to write_dynamic_frame.from_options
s3_sink = glueContext.getSink(
    connection_type="s3",
    path="s3://my-bucket/silver/orders/",
    enableUpdateCatalog=False
)
s3_sink.setFormat("parquet")
s3_sink.writeFrame(dyf_from_rdd)

In [ ]:
# Write to S3 AND update Glue Catalog at the same time
# This is the key advantage of getSink over write_dynamic_frame.from_options
catalog_sink = glueContext.getSink(
    connection_type="s3",
    path="s3://my-bucket/silver/orders/",
    enableUpdateCatalog=True,          # updates Glue Catalog after write
    updateBehavior="UPDATE_IN_DATABASE" # options: UPDATE_IN_DATABASE, LOG
)
catalog_sink.setCatalogInfo(
    catalogDatabase="ecommerce_db",
    catalogTableName="orders_silver"
)
catalog_sink.setFormat("parquet")
catalog_sink.writeFrame(dyf_from_rdd)

**What `enableUpdateCatalog=True` does:**
```
Without enableUpdateCatalog:
  → Data written to S3
  → Glue Catalog NOT updated
  → Athena cannot see the new data until you run a crawler

With enableUpdateCatalog=True:
  → Data written to S3
  → Glue Catalog updated automatically
  → Athena can query the new data immediately
```

### getSink vs write_dynamic_frame — which to use?

| | `write_dynamic_frame.from_options` | `getSink().writeFrame()` |
|---|---|---|
| Style | Concise, one call | Verbose, two steps |
| Write to S3 | ✅ | ✅ |
| Update Glue Catalog on write | ❌ | ✅ via `enableUpdateCatalog=True` |
| Auto-generated by Glue Visual Editor | No | Yes |
| Recommended for new code | ✅ | Only when you need `enableUpdateCatalog` |


---
# Final Comparison